# Presenter setup (not a slide)

English · 11 slides · about 10 minutes. Source: `01_data_exploration.ipynb`, H2–H4 and H8–H14, plus profile context from section 4.1. Original rows are unchanged.

Run all cells before presenting. The repository CSV is read from `data/credit_default.csv` (two header lines). Without it, clearly identified saved summaries are used: rounded demographic rates, limit endpoints and approximate age ranges, not simulated clients. Charts were tested in saved-summary mode. The CSV path and charts were tested on the five actual example rows displayed in the source notebook, not on the full CSV. Outputs are cleared for version control.

Install `pandas ipykernel jupyterlab-rise` for JupyterLab/Notebook 7; classic Notebook 6 uses `rise`. Trust the notebook, run all cells, then launch the RISE toolbar control. Setup is skipped, code inputs hidden, speaker notes assigned. https://rise.readthedocs.io/ and https://github.com/jupyterlab-contrib/rise

Provenance: Yeh & Lien (2009), UCI Default of Credit Card Clients, as described in the source. April–September 2005 history; October 2005 outcome. NT$ = New Taiwan dollars. No external images or fonts required.


In [ ]:
from pathlib import Path
from html import escape
import math
import pandas as pd
from IPython.display import SVG, HTML, display

NAVY, TEAL, CORAL, MUTED = '#183247', '#168b8a', '#d76050', '#687b89'
# Rounded fallback values are transcribed from saved results, not reconstructed rows.
S = {'n':30000, 'columns':25, 'missing':0, 'unique':True, 'repeats':35,
     'profile_repeats':56, 'profile_rows':108, 'profile_groups':52, 'mixed_groups':21,
     'defaults':6636, 'limit':[10000,50000,140000,240000,1000000],
     'age':[21,28,34,41,79], 'bill':[22381.5,350110.68,964511],
     'payment':[2100,66522.18,873552],
     'zero_bill':[13.40,11.69,10.65,9.57,8.35,6.69],
     'zero_payment':[23.91,22.34,21.36,19.89,17.99,17.50],
     'demographics':[['Sex: men',.242],['Sex: women',.208],
       ['Education: graduate school',.192],['Education: university',.237],
       ['Education: high school',.252],['Marital status: married',.235],
       ['Marital status: single',.209]],
     'limit_rates':[['Lowest limit group (up to NT$30k)',.358],
                    ['Highest limit group (above NT$360k)',.119]], 'age_rates':None}
paths = [Path('data/credit_default.csv'), Path('../data/credit_default.csv')]
path = next((p for p in paths if p.exists()), None)
MODE = 'Saved notebook results; rounded where reported'
if path is not None:
    df = pd.read_csv(path, header=1).rename(columns={'default payment next month':'DEFAULT'})
    required = {'ID','DEFAULT','LIMIT_BAL','AGE','SEX','EDUCATION','MARRIAGE'} | {f'{prefix}{i}' for prefix in ['BILL_AMT','PAY_AMT'] for i in range(1,7)}
    if not required.issubset(df.columns) or not df.DEFAULT.isin([0,1]).all():
        raise ValueError('Unexpected dataset schema or outcome values; no silent fallback.')
    profiles = df.drop(columns=['ID','DEFAULT'])
    groups = df.loc[profiles.duplicated(keep=False)].groupby(list(profiles.columns)).DEFAULT.agg(['size','nunique'])
    S.update(n=len(df),columns=len(df.columns),missing=int(df.isna().sum().sum()),
        unique=bool(df.ID.is_unique),repeats=int(df.drop(columns='ID').duplicated().sum()),
        profile_repeats=int(profiles.duplicated().sum()),profile_rows=int(groups['size'].sum()),
        profile_groups=len(groups),mixed_groups=int((groups['nunique']>1).sum()),defaults=int(df.DEFAULT.sum()))
    for key,col in [('limit','LIMIT_BAL'),('age','AGE')]:
        S[key]=df[col].quantile([0,.25,.5,.75,1]).tolist()
    for key,col in [('bill','BILL_AMT1'),('payment','PAY_AMT1')]:
        S[key]=df[col].quantile([.5,.99,1]).tolist()
    for key,prefix in [('zero_bill','BILL_AMT'),('zero_payment','PAY_AMT')]:
        S[key]=[(df[f'{prefix}{i}']==0).mean()*100 for i in range(6,0,-1)]
    def rates(series):
        g=df.groupby(series,observed=True).DEFAULT.agg(['sum','size'])
        return [[str(k),r['sum']/r['size'],int(r['size']),int(r['sum'])] for k,r in g.iterrows()]
    S['demographics']=[]
    for col,mapping in [('SEX',{1:'Sex: men',2:'Sex: women'}),
                        ('EDUCATION',{1:'Education: graduate school',2:'Education: university',3:'Education: high school'}),
                        ('MARRIAGE',{1:'Marital status: married',2:'Marital status: single'})]:
        for label,rate,n,k in rates(df[col]):
            if int(label) in mapping: S['demographics'].append([mapping[int(label)],rate,n,k])
    S['limit_rates']=rates(pd.qcut(df.LIMIT_BAL,10,duplicates='drop'))
    S['age_rates']=rates(pd.cut(df.AGE,[20,25,30,35,40,45,50,60,80]))
    MODE='Recomputed from repository CSV; original rows unchanged'
BASE = S['defaults']/S['n']

def text(x,y,s,size=19,color=NAVY):
    return f'<text x="{x}" y="{y}" fill="{color}" font-size="{size}">{escape(str(s))}</text>'
def line(x1,y1,x2,y2,color=MUTED,extra=''):
    return f'<line x1="{x1}" y1="{y1}" x2="{x2}" y2="{y2}" stroke="{color}" {extra}/>'
def render(parts,height=360):
    display(SVG('<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1000 '+str(height)+'" role="img" style="font-family:Arial,sans-serif;background:white">'+''.join(parts)+'</svg>'))
def cards(items):
    parts=[]; w=960/len(items)
    for i,(value,label) in enumerate(items):
        x=20+i*w
        parts += [f'<rect x="{x}" y="25" width="{w-16}" height="150" rx="12" fill="#eef5f6"/>',text(x+18,90,value,38,TEAL),text(x+18,135,label,18)]
    render(parts,200)
def bars(rows,maximum=.4,reference=None):
    left,right=400,870; h=75+42*len(rows); parts=[]
    for tick in range(5):
        v=maximum*tick/4; x=left+(right-left)*v/maximum
        parts += [line(x,20,x,h-40,'#e3eaee'),text(x-14,h-12,f'{v:.0%}',16,MUTED)]
    if reference is not None:
        x=left+(right-left)*reference/maximum
        parts += [line(x,12,x,h-40,NAVY,'stroke-dasharray="5 5"'),text(650,14,f'Overall: {reference:.1%}',16)]
    for i,row in enumerate(rows):
        label,value=row[:2]; y=45+i*42; x=left+(right-left)*value/maximum
        parts += [text(8,y+5,label,18),f'<rect x="{left}" y="{y-15}" width="{x-left}" height="23" rx="3" fill="{CORAL}"/>',text(x+9,y+4,f'{value:.1%}',18)]
        if len(row)==4:
            n,k=row[2:]; z=1.96; d=1+z*z/n; c=(value+z*z/(2*n))/d
            half=z*math.sqrt(value*(1-value)/n+z*z/(4*n*n))/d
            parts += [line(left+(right-left)*(c-half)/maximum,y-3,left+(right-left)*(c+half)/maximum,y-3,NAVY,'stroke-width="2"')]
    render(parts,h)

display(HTML('<style>\n.reveal,.rise-enabled{background:#fff!important;color:#183247!important;font-family:Arial,sans-serif!important}\n.reveal h1,.reveal h2,.reveal h3{color:#183247!important;font-family:Arial,sans-serif!important;text-transform:none!important;text-align:left!important}\n.reveal h1{font-size:2em!important}.reveal h2{font-size:1.35em!important}\n.reveal p,.reveal li{font-size:24px!important;line-height:1.35!important;text-align:left!important}\n.reveal .input,.reveal .prompt,.reveal .jp-InputArea,.rise-enabled .jp-InputArea,.rise-enabled .jp-InputPrompt,.rise-enabled .jp-OutputPrompt{display:none!important}\n.reveal svg,.rise-enabled svg{max-width:100%;max-height:430px}\n.reveal .output_area,.reveal .output_subarea{padding:0!important}\n</style>'))

print(MODE)


## Understanding the credit default database

A visual tour of 30,000 credit-card records

Taiwan · April–September 2005 history · October 2005 outcome

In [ ]:
cards([('30,000','records'),('6 months','account history'),('1 outcome','default: yes / no')])

One Taiwanese bank, 2005: this sample is not all borrowers or current customers. Descriptive analysis only. Source: 01_data_exploration.ipynb introduction.

## What is inside one record?

25 columns = 1 identifier + 23 descriptive variables + 1 outcome. All money amounts are in New Taiwan dollars (NT$).

In [ ]:
cards([('5','profile + credit limit'),('18','6 statuses + 6 bills + 6 payments'),('1','October outcome')])

Five profile/limit variables: credit limit, sex, education, marital status, age. ID is separate. Six monthly records per client are not independent clients. History April–September precedes the October default flag.

## H2–H4 | Complete rows, but repeated profiles

H2: no missing cells. H3: each identifier is unique. H4 needs a more careful answer: different IDs can have identical recorded information.

In [ ]:
cards([(f"{S['missing']:,}",'missing cells'),('Yes' if S['unique'] else 'No','unique identifiers'),(str(S['repeats']),'repeat rows beyond the first')])
print(f"Profile check: {S['profile_rows']} rows in {S['profile_groups']} identical-profile groups; {S['mixed_groups']} groups have both outcomes.")

35 counts repeats beyond first occurrences, excluding ID but including outcome. Excluding both ID and outcome gives 56 repeats, in 52 groups/108 rows; 21 groups contain both outcomes. Repeated profiles do not prove duplicate people. No rows removed.

## H8 | About 1 in 5 clients defaulted

Default rate = clients with the default flag ÷ all clients. No prediction is involved.

In [ ]:
bars([['No default',1-BASE],['Default',BASE]],maximum=1)
print(f"{S['defaults']:,} of {S['n']:,} records have default = 1 ({BASE:.2%}).")

6,636 default and 23,364 no-default records. No-default does not mean the whole balance was paid. Later charts use the same overall baseline.

## A typical client is not the average of every client

Additional context: the middle half of the sample is aged 28–41 and has credit limits of NT$50k–240k.

In [ ]:
cards([(f"{S['age'][2]:.0f} years",'median age'),(f"NT${S['limit'][2]/1000:.0f}k",'median credit limit'),('28–41','middle-half age range')])

Median: half the records on each side. Middle half: 25th–75th percentiles. Ages span 21–79; limits NT$10k–1m. Source section 4.1.

## H9 | A few amounts are much larger than most

September amounts: compare the median, the 99th percentile and the largest recorded value. The horizontal axis is logarithmic: equal steps mean ×10.

In [ ]:
parts=[]
for v in [1000,10000,100000,1000000]:
 x=330+170*math.log10(v/1000)
 parts += [line(x,20,x,210,'#e3eaee'),text(x-20,245,f'{v/1000:g}k',17)]
for y,key,label in [(65,'bill','September bill'),(160,'payment','September payment')]:
 values=S[key]; xs=[330+170*math.log10(v/1000) for v in values]
 parts += [text(5,y,label),line(xs[0],y,xs[-1],y,TEAL,'stroke-width="3"')]
 for x,v,name in zip(xs,values,['Median','99th pct.','Maximum']):
  parts += [f'<circle cx="{x}" cy="{y}" r="6" fill="{TEAL}"/>',text(x-25,y-16,f'{v/1000:.1f}k',16),text(x-30,y+27,name,14,MUTED)]
parts += [text(660,280,'Amount (NT$, log scale)',18)]
render(parts,300)

Bill median/99th percentile/max: 22,381.5/350,110.68/964,511. Payments: 2,100/66,522.18/873,552. Positive markers on a log axis omit zero/negative values visually, not from the data. Equal axis steps mean multiplication by 10.

## H9 | Zero is a value, not an empty cell

Additional descriptive view: the percentage of zero payments is higher than the percentage of zero bills in every observed month.

In [ ]:
parts=[]
for i,month in enumerate(['Apr','May','Jun','Jul','Aug','Sep']):
 x=80+i*150
 parts += [text(x,245,month,20)]
 for dx,key,color in [(0,'zero_bill',TEAL),(46,'zero_payment',NAVY)]:
  v=S[key][i]; h=v*6
  parts += [f'<rect x="{x+dx}" y="{205-h}" width="32" height="{h}" fill="{color}" rx="3"/>',text(x+dx-3,195-h,f'{v:.1f}',14,color)]
parts += [text(60,285,'Teal: zero bill     Navy: zero payment     Labels: % of records',18)]
render(parts,310)

Common zero-based bar scale. Months run April–September. 6.4% have at least one negative bill; this may reflect credit balances/refunds, not proven individual causes. Zero payment does not by itself prove a missed obligation.

## H10–H12 | Demographic gaps are modest

The main education levels differ more than sex or marital status. These are separate comparisons, not explanations of why default happens.

In [ ]:
bars(S['demographics'],maximum=.4,reference=BASE)

H10/H11/H12 are separate, unadjusted associations. Main documented categories only: all other codes remain in the data and baseline. Rounded saved rates have no intervals; CSV reruns add 95% Wilson intervals. Small gaps are not individual explanations.

## H13 | Lower credit limits go with more default

The source reports 35.8% in the lowest limit group and 11.9% in the highest. Raising a limit is not shown to reduce default.

In [ ]:
bars(S['limit_rates'],maximum=.5,reference=BASE)

Saved snapshot shows only two reported endpoints. CSV rerun shows all qcut groups with Wilson intervals. Tied limits mean groups need not be equal-sized; duplicate boundaries are dropped. Association does not establish that changing a limit changes default.

## H14 | Default does not fall steadily with age

The lowest rates are around ages 26–35; they are higher among the youngest and older groups. The source therefore rejects a steadily decreasing pattern.

In [ ]:
if S['age_rates'] is None:
    cards([('26.7%','ages 21–25'),('about 19–20%','ages 26–35'),('about 25–27%','groups above 45')])
else:
    bars(S['age_rates'],maximum=.5,reference=BASE)

Saved age cards show rounded ranges over several bins, not pooled-group estimates or invented exact rates. CSV rerun shows all eight original age bins with Wilson intervals. Older groups are smaller. Similar medians can hide different group-rate patterns.

## What should the audience remember?

1. The table is complete, but some recorded profiles repeat.
2. About 22% of clients have the default flag.
3. Monetary values have long tails and many zeros.
4. Demographic differences are modest; credit limit and age show clearer patterns.
5. These observations describe one bank’s historical sample—not causes or predictions.

No cleaning recommendations, models, feature ranking or prediction claims. All findings describe the source sample.